h1. Setting Up Requirements

In [ ]:
!pip install llama-index-workflows llama-cloud-services jsonschema openai neo4j llama-index-llms-openai

In [ ]:
import re
import uuid
import os
from datetime import date
from typing import List, Optional

from getpass import getpass
from neo4j import AsyncGraphDatabase
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

from llama_cloud_services.beta.classifier.client import ClassifyClient
from llama_cloud.types import ClassifierRule
from llama_cloud.client import AsyncLlamaCloud

from llama_cloud_services.extract import (
    ExtractConfig,
    ExtractMode,
    LlamaExtract,
    SourceText,
)
from llama_cloud_services.parse import LlamaParse

from llama_index.llms.openai import OpenAI

Download Sample Contract

In [3]:
!wget https://raw.githubusercontent.com/tomasonjo/blog-datasets/5e3939d10216b7663687217c1646c30eb921d92f/CybergyHoldingsInc_Affliate%20Agreement.pdf

--2026-02-22 13:21:25--  https://raw.githubusercontent.com/tomasonjo/blog-datasets/5e3939d10216b7663687217c1646c30eb921d92f/CybergyHoldingsInc_Affliate%20Agreement.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 134300 (131K) [application/octet-stream]
Saving to: ‘CybergyHoldingsInc_Affliate Agreement.pdf’

CybergyHoldingsInc_ 100%[===================>] 131.15K  --.-KB/s    in 0.08s   

2026-02-22 13:21:25 (1.54 MB/s) - ‘CybergyHoldingsInc_Affliate Agreement.pdf’ saved [134300/134300]



Set up Neo4j

In [ ]:
import os

db_url = os.getenv("NEO4J_DB_URL")
username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

neo4j_driver = AsyncGraphDatabase.driver(
    db_url,
    auth=(
        username,
        password,
    ),
)

In [6]:
from neo4j import GraphDatabase
AUTH=(username, password)
with GraphDatabase.driver(db_url, auth=AUTH) as driver:
    driver.verify_connectivity()

Parse the Contract with LlamaParse

In [7]:
import os

os.environ["LLAMA_CLOUD_API_KEY"] = getpass("Enter your Llama API key: ")
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [8]:
# Initialize parser with specified mode
parser = LlamaParse(parse_mode="parse_page_without_llm")

# Define the PDF file to parse
pdf_path = "CybergyHoldingsInc_Affliate Agreement.pdf"

# Parse the document asynchronously
results = await parser.aparse(pdf_path)

Started parsing the file under job_id a2b9a51f-53b0-4917-bf7a-36cc0c851130


In [9]:
print(results.pages[0].text)

Exhibit 10.27

MARKETING AFFILIATE AGREEMENT

Between:

Birch First Global Investments Inc.

And

Mount Knowledge Holdings Inc.

Dated: May 8, 2014

1

Source: CYBERGY HOLDINGS, INC., 10-Q, 5/20/2014


Contract classification with LlamaClassify

In [10]:
#Define classification rules
rules = [
    ClassifierRule(
        type="Affiliate Agreements",
        description="This is a contract that outlines an affiliate agreement",
    ),
    ClassifierRule(
        type="Co Branding",
        description="This is a contract that outlines a co-branding deal/agreement",
    ),
]

In [11]:
classifier = ClassifyClient(
    client=AsyncLlamaCloud(
        base_url="https://api.cloud.llamaindex.ai",
        token=os.environ["LLAMA_CLOUD_API_KEY"],
    )
)

In [13]:
result = await classifier.aclassify_file_path(
    rules=rules,
    file_input_path="CybergyHoldingsInc_Affliate Agreement.pdf",
)

/tmp/ipykernel_13826/513466347.py:1: DeprecationWarning: aclassify_file_path is deprecated, use aclassify() instead
  result = await classifier.aclassify_file_path(


In [14]:
classification = result.items[0].result
print("Classification Result: " + classification.type)
print("Classification Reason: " + classification.reasoning)

Classification Result: Affiliate Agreements
Classification Reason: This document is clearly titled "MARKETING AFFILIATE AGREEMENT" and is between Birch First Global Investments Inc. (Company) and Mount Knowledge Holdings Inc. (Marketing Affiliate/MA). The document explicitly establishes an affiliate relationship where the Marketing Affiliate is granted rights to advertise, market, and sell the Company's Technology to clients. Key indicators include: (1) The document is explicitly named as an "Affiliate Agreement" both in the filename and document title, (2) It outlines the duties of the Marketing Affiliate including marketing, sales, support, and reporting obligations, (3) It establishes commission/pricing structures and quotas typical of affiliate arrangements, (4) It grants distribution and marketing rights rather than joint branding rights, (5) There is no mention of co-branding, joint marketing materials, or shared brand identity which would be characteristic of a Co Branding agree

Setting Up LlamaExtract

In [15]:
class Location(BaseModel):
    """Location information with structured address components."""

    country: Optional[str] = Field(None, description="Country")
    state: Optional[str] = Field(None, description="State or province")
    address: Optional[str] = Field(None, description="Street address or city")


class Party(BaseModel):
    """Party information with name and location."""

    name: str = Field(description="Party name")
    location: Optional[Location] = Field(
        None, description="Party location details"
    )

In [16]:
class BaseContract(BaseModel):
    """Base contract class with common fields."""

    parties: Optional[List[Party]] = Field(
        None, description="All contracting parties"
    )
    agreement_date: Optional[str] = Field(
        None, description="Contract signing date. Use YYYY-MM-DD"
    )
    effective_date: Optional[str] = Field(
        None, description="When contract becomes effective. Use YYYY-MM-DD"
    )
    expiration_date: Optional[str] = Field(
        None, description="Contract expiration date. Use YYYY-MM-DD"
    )
    governing_law: Optional[str] = Field(
        None, description="Governing jurisdiction"
    )
    termination_for_convenience: Optional[bool] = Field(
        None, description="Can terminate without cause"
    )
    anti_assignment: Optional[bool] = Field(
        None, description="Restricts assignment to third parties"
    )
    cap_on_liability: Optional[str] = Field(
        None, description="Liability limit amount"
    )


class AffiliateAgreement(BaseContract):
    """Affiliate Agreement extraction."""

    exclusivity: Optional[str] = Field(
        None, description="Exclusive territory or market rights"
    )
    non_compete: Optional[str] = Field(
        None, description="Non-compete restrictions"
    )
    revenue_profit_sharing: Optional[str] = Field(
        None, description="Commission or revenue split"
    )
    minimum_commitment: Optional[str] = Field(
        None, description="Minimum sales targets"
    )


class CoBrandingAgreement(BaseContract):
    """Co-Branding Agreement extraction."""

    exclusivity: Optional[str] = Field(
        None, description="Exclusive co-branding rights"
    )
    ip_ownership_assignment: Optional[str] = Field(
        None, description="IP ownership allocation"
    )
    license_grant: Optional[str] = Field(
        None, description="Brand/trademark licenses"
    )
    revenue_profit_sharing: Optional[str] = Field(
        None, description="Revenue sharing terms"
    )


mapping = {
    "affiliate_agreements": AffiliateAgreement,
    "co_branding": CoBrandingAgreement,
}

In [19]:
def normalize_contract_type(value: str) -> str:
    return value.strip().lower().replace(" ", "_")

extractor = LlamaExtract()

normalized_type = normalize_contract_type(classification.type)

schema = mapping.get(normalized_type)

if not schema:
    raise ValueError(f"Unsupported contract type: {classification.type}")

agent = extractor.create_agent(
    name=f"extraction_workflow_import_{uuid.uuid4()}",
    data_schema=schema,
    config=ExtractConfig(
        extraction_mode=ExtractMode.BALANCED,
    ),
)

result = await agent.aextract(
    files=SourceText(
        text_content=" ".join([el.text for el in results.pages]),
        filename=pdf_path,
    ),
)

result.data

Extracting files: 100%|██████████| 1/1 [00:24<00:00, 24.85s/it]


{'parties': [{'name': 'Birch First Global Investments Inc.',
   'location': {'country': 'U.S. Virgin Islands',
    'state': None,
    'address': '9100 Havensight, Port of Sale, Ste. 15/16, St. Thomas, VI 0080'}},
  {'name': 'Mount Knowledge Holdings Inc.',
   'location': {'country': 'United States',
    'state': 'Nevada',
    'address': '228 Park Avenue S. #56101 New York, NY 100031502'}}],
 'agreement_date': '2014-05-08',
 'effective_date': '2014-05-08',
 'expiration_date': None,
 'governing_law': 'State of Nevada',
 'termination_for_convenience': True,
 'anti_assignment': True,
 'cap_on_liability': "Company's liability shall not exceed the fees that MA has paid under this Agreement.",
 'exclusivity': None,
 'non_compete': None,
 'revenue_profit_sharing': 'MA receives a purchase discount: 15% ($0-$100,000), 20% ($100,001-$1,000,000), 25% ($1,000,001 and above) from the Company, based on annual purchase levels.',
 'minimum_commitment': 'MA commits to purchase a minimum of 100 Units wit

Import into neo4j

In [20]:
import_query = """
WITH $contract AS contract
MERGE (c:Contract {path: $path})
SET c += apoc.map.clean(contract, ["parties", "agreement_date", "effective_date", "expiration_date"], [])
// Cast to date
SET c.agreement_date = date(contract.agreement_date),
    c.effective_date = date(contract.effective_date),
    c.expiration_date = date(contract.expiration_date)

// Create parties with their locations
WITH c, contract
UNWIND coalesce(contract.parties, []) AS party
MERGE (p:Party {name: party.name})
MERGE (c)-[:HAS_PARTY]->(p)

// Create location nodes and link to parties
WITH p, party
WHERE party.location IS NOT NULL
MERGE (p)-[:HAS_LOCATION]->(l:Location)
SET l += party.location
"""

response = await neo4j_driver.execute_query(
    import_query, contract=result.data, path=pdf_path
)
response.summary.counters

SummaryCounters({'contains-updates': True, 'labels-added': 5, 'relationships-created': 4, 'nodes-created': 5, 'properties-set': 16})